# 🔬 Feature Lab – Quant Researcher
**Workflow**: Load OHLCV → Compute features → Interactive EDA → Feature importance → Correlation Analysis → Save to FeatureStore

---

In [4]:
from qtrader.output.analyst import AnalystSession, RoleContext
from qtrader.input.features.store import FeatureStore
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

session = AnalystSession(role=RoleContext.RESEARCHER)
session.info()

SYMBOL = 'BTC-USD'
TIMEFRAME = '1d'

🔬 Quant Researcher Workflow
  1. load_ohlcv → raw OHLCV
  2. add_rolling_features → compute & store features via FeatureStore
  3. run_alpha_score → forward-return scoring
  4. Use MLflow for experiment tracking (see notebooks/researcher/)
  5. load_features → pull pre-computed features

  📓 Notebooks: notebooks/researcher/01_Feature_Lab.ipynb, ...



## 1. Data Ingestion & Engineering
Load market data and augment with technical factors.

In [5]:
try:
    df = session.load_ohlcv(SYMBOL, TIMEFRAME)
except Exception:
    df = session.sample_ohlcv(symbol='BTC', days=730)

df = session.make_returns(df)
df = session.add_rolling_features(df, windows=[5, 10, 21, 63])
df = df.drop_nulls()

# Alpha scores (forward returns)
df = session.run_alpha_score(df, forward_periods=[1, 5, 10])
df = df.drop_nulls()

print(f"Dataset shape: {df.shape}")
df.head()

DuckDB load failed (No datalake partition for BTC-USD 1d at data_lake/symbol=BTC-USD/tf=1d). Falling back to datalake.
UniversalDataLake missing (No such file or directory (os error 2): ./data_lake/symbol=BTC-USD/tf=1d/data.parquet). Falling back to DataLake.
DataLake missing. Falling back to Live API.
Failed to persist live data to DataLake: [Errno 2] Failed to open local file './data_lake/symbol=BTC-USD/tf=1d/data.parquet'. Detail: [errno 2] No such file or directory


Dataset shape: (0, 22)


timestamp,open,high,low,close,volume,returns,sma_5,vol_5,sma_10,vol_10,sma_21,vol_21,sma_63,vol_63,avg_gain,avg_loss,rsi_14,fwd_ret_1,fwd_ret_5,fwd_ret_10,alpha_score
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


## 2. Interactive Correlation Analysis
Examine how features correlate with forward returns across different horizons.

In [10]:
feature_cols = [c for c in df.columns if c.startswith(('sma_', 'vol_', 'rsi_', 'avg_'))]
target_cols  = ['fwd_ret_1', 'fwd_ret_5', 'fwd_ret_10']

corr_matrix = []
for fc in feature_cols:
    row = []
    for tc in target_cols:
        c = np.corrcoef(df[fc], df[tc])[0, 1]
        row.append(c)
    corr_matrix.append(row)

fig = px.imshow(
    corr_matrix, 
    x=target_cols, 
    y=feature_cols, 
    color_continuous_scale='RdBu_r', 
    aspect="auto",
    labels=dict(color="Correlation"),
    title="Feature × Forward Return Correlation Matrix"
)
fig.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    width=800, 
    height=max(400, len(feature_cols)*25)
)
fig.show()

/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:552: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/numpy/_core/_methods.py:137: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2894: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/hoangnam/qtrader/.venv/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:2894: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


## 3. Dynamic Feature Stability
Check if the predictive power of the top feature is stable over time.

In [7]:
corr_vals = np.array(corr_matrix)
top_idx = np.unravel_index(np.argmax(np.abs(corr_vals)), corr_vals.shape)
top_feat = feature_cols[top_idx[0]]
target_feat = target_cols[top_idx[1]]

print(f"Analyzing Stability for: {top_feat} vs {target_feat}")

win = 90
timestamps = df['timestamp'].to_numpy()[win:]
feat_vals = df[top_feat].to_numpy()
target_vals = df[target_feat].to_numpy()

rolling_corr = [np.corrcoef(feat_vals[i:i+win], target_vals[i:i+win])[0,1] for i in range(len(feat_vals)-win)]

fig_stab = go.Figure()
fig_stab.add_trace(go.Scatter(
    x=timestamps, 
    y=rolling_corr, 
    mode='lines', 
    name=f'Rolling {win}d Corr',
    line=dict(color='#a78bfa', width=2)
))
fig_stab.add_hline(y=0, line_dash="dash", line_color="#64748b")
fig_stab.update_layout(
    title=f"Predictive Stability: {top_feat} vs {target_feat}",
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117',
    xaxis_title="Date",
    yaxis_title="Correlation",
    height=400
)
fig_stab.show()

Analyzing Stability for: sma_5 vs fwd_ret_1


## 4. Feature Distribution & Outliers
Understanding feature behavior across different market conditions.

In [8]:
fig_dist = px.histogram(
    df.to_pandas(), 
    x=top_feat, 
    marginal="box", 
    title=f"Distribution of {top_feat}", 
    color_discrete_sequence=['#34d399']
)
fig_dist.update_layout(
    template="plotly_dark",
    plot_bgcolor='#0f1117',
    paper_bgcolor='#0f1117'
)
fig_dist.show()

## 5. Persistence
Save engineered features to the unified store.

In [9]:
store = FeatureStore()
save_cols = ['timestamp', 'close', 'returns'] + feature_cols + target_cols
store.save_features(df.select(save_cols), symbol=SYMBOL, timeframe=TIMEFRAME)
print(f"✅ Features persisted to store for {SYMBOL}/{TIMEFRAME}")

✅ Features persisted to store for BTC-USD/1d
